# Import libraries and set variables

In [ ]:
import os
import platform
import sys
from pathlib import Path
from time import sleep

import pandas as pd
from tqdm import tqdm

tqdmBarFormat = "{l_bar}{bar:10}| {n_fmt}/{total_fmt} [Elapsed:{elapsed}<Remaining:{remaining}{rate_fmt}{postfix}]"

osName = platform.system()
if osName == 'Windows':
    dataverseFunctionsPath = os.path.join(
        'C:\\', 'Users', 'Owner', 'Documents', 'GitHub', 'dataverse-functions')
elif osName == 'Darwin':
    dataverseFunctionsPath = os.path.join(
        '/Users', 'juliangautier', 'dataverse-functions')
sys.path.append(dataverseFunctionsPath)  
from dataverse_functions import common_functions, dataverse_functions

desktopPath = os.path.join(Path.home(), 'Desktop')

# Enter a user agent and your email address. Some Dataverse installations block requests from scripts.
# See https://www.whatismybrowser.com/detect/what-is-my-user-agent to get your user agent
userAgent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/81.0.4044.138 Safari/537.36'
emailAddress = 'juliangautier@g.harvard.edu'

headers = {
    'User-Agent': userAgent,
    'From': emailAddress}

rorRegex = r'https?://ror\.org/0[a-hj-km-np-tv-z|0-9]{6}[0-9]{2}$'
processedDataDir = '/Users/juliangautier/ux-research-data-analysis/Findability_of_author_affiliations_with_rors/data/processed'

# Measure bug's affect on findability of datasets in Dataverse installations

In [ ]:
# Get names and ROR URLs of author affiliations in datasets published in each Dataverse installation
rorUrlPrenset_affNameNotPresentDf = pd.read_csv(
    os.path.join(processedDataDir, 'rorUrlPrenset_affNameNotPresent.csv'),
    encoding='utf-8-sig')

rorUrlPrenset_affNameNotPresentDf.head()

In [28]:
installationsToIgnore = [
    'BSC Dataverse',
    'CORA. Research Data Repository (RDR)',
    'DANS Data Station Archaeology',
    'DANS Data Station Life Sciences',
    'DANS Data Station Physical and Technical Sciences',
    'DaRUS',
    'DataverseNL',
    'DataverseNO',
    'Harvard Dataverse',
    'MBLWHOI Library Dataverse',
    'Texas Data Repository Dataverse',
    'TUDOdata'
]

In [ ]:
# For each affiliation name, do a name-based search and a ROR-based search for each ROR

loopObj = tqdm(
    bar_format=tqdmBarFormat,
    # total=len(dataframe),
    iterable=rorUrlPrenset_affNameNotPresentDf.groupby(['dataverse_installation_url']))

for _, installationDf in loopObj:

# for _, installationDf in rorUrlPrenset_affNameNotPresentDf.groupby(['dataverse_installation_url']):
    installationName = installationDf['dataverse_installation_name'].iloc[0]
    installation_url = installationDf['dataverse_installation_url'].iloc[0]
    safeInstallationName = common_functions.safe_filename(installationName)
    loopObj.set_postfix_str(installationName)
    
    if installationName not in installationsToIgnore:

        # Create directories for affiliaton search results
        installationDirPath = os.path.join(desktopPath, 'affiliation_search_results', safeInstallationName)
        os.makedirs(installationDirPath)

        searchResultComparisonDict = []

        rootCollectionInfoDict = dataverse_functions.get_root_collection_info(installation_url)
        rootAlias = rootCollectionInfoDict['rootAlias']
        totalAffiliationCount = len(installationDf)
        for dfIndex, dfRow in installationDf.iterrows():
            authorAffiliationName = dfRow['ror_display_name_2026_11_01']
            loopObj.set_postfix_str(f'{installationName}: {authorAffiliationName}')
            # print(f'Searching for affiliation {progressCount} of {totalAffiliationCount} by name and ROR URL: {authorAffiliationName}')

            safeAffiliationName = common_functions.safe_filename(authorAffiliationName)
            affiliationDirPath = os.path.join(installationDirPath, 'affiliation_search_results', safeAffiliationName)
            os.makedirs(affiliationDirPath)

            authorAffiliationRorUrlList = common_functions.string_to_list(dfRow['author_affiliation_ror'])
            
            # Use Search API to get list of datasets returned by search of author affiliation name
            nameBasedSearchUrl = f'{installation_url}/dataverse/{rootAlias}/?q=authorAffiliation:"{authorAffiliationName}"&fq1=publicationStatus:"Published"&types=datasets'

            sleep(.5)

            try:
                nameBasedSearchResultsDf = dataverse_functions.get_datasets_from_collection_or_search_url(
                    nameBasedSearchUrl, headers=headers, apiKey='', 
                    metadataFieldsParam=['citation:author'],
                    ignoreDeaccessionedDatasets=True,
                    includeSubDataverses=True, includeLinkedDatasets=True, 
                    includeHarvestedDatasets=False,
                    printProgress=False)
            except Exception:
                nameBasedSearchResultsDf = None

            if nameBasedSearchResultsDf is None:
                datasetPidsFromNameBasedSearch = []

            if nameBasedSearchResultsDf is not None:

                # Reorder columns
                nameBasedSearchResultsDf = nameBasedSearchResultsDf[[
                    'dataset_pid', 'dataverse_collection_alias', 
                    'latest_version', 'dataset_last_updated', 'citation:author'
                ]]

                datasetPidsFromNameBasedSearch = nameBasedSearchResultsDf['dataset_pid'].to_list()
                datasetPidsFromNameBasedSearch = list(set(datasetPidsFromNameBasedSearch))

                # Export search results
                filePath = os.path.join(affiliationDirPath,  'name_based_search_results.csv')
                nameBasedSearchResultsDf.to_csv(filePath, index=False,  encoding='utf-8-sig')
            
            sleep(.5)

            # Use Search API to get list of datasets returned by search of ROR URLs
            rorBasedSearchResultsDfList = []
            for rorUrl in authorAffiliationRorUrlList:
                rorBasedSearchUrl = f'{installation_url}/dataverse/{rootAlias}/?q=authorAffiliation:"{rorUrl}"&fq1=publicationStatus:"Published"&types=datasets'

                try:
                    rorBasedSearchResultsDf = dataverse_functions.get_datasets_from_collection_or_search_url(
                        rorBasedSearchUrl, headers=headers, apiKey='', 
                        metadataFieldsParam=['citation:author'],
                        ignoreDeaccessionedDatasets=True,
                        includeSubDataverses=True, includeLinkedDatasets=True, 
                        includeHarvestedDatasets=False,
                        printProgress=False)
                except Exception:
                    rorBasedSearchResultsDf = None

                if rorBasedSearchResultsDf is not None:
                    rorBasedSearchResultsDfList.append(rorBasedSearchResultsDf)


            if len(rorBasedSearchResultsDfList) == 0:
                datasetPidsFromRorBasedSearch = []

            if len(rorBasedSearchResultsDfList) > 0:
                # Concat dataframes in rorBasedSearchResultsDfList
                rorBasedSearchDf = pd.concat(rorBasedSearchResultsDfList)

                # Reorder columns
                rorBasedSearchResultsDf = rorBasedSearchResultsDf[[
                    'dataset_pid', 'dataverse_collection_alias', 
                    'latest_version', 'dataset_last_updated', 'citation:author'
                ]]

                datasetPidsFromRorBasedSearch = rorBasedSearchResultsDf['dataset_pid'].to_list()
                datasetPidsFromRorBasedSearch = list(set(datasetPidsFromRorBasedSearch))

                # Export search results
                filePath = os.path.join(affiliationDirPath,  'ror_based_search_results.csv')
                rorBasedSearchResultsDf.to_csv(filePath, index=False,  encoding='utf-8-sig')

            # If all pids in datasetPidsFromRorBasedSearch are in datasetPidsFromNameBasedSearch
            nameBasedSearchItemsIncludeAllRorBasedSearchItems = all(
                pid in datasetPidsFromNameBasedSearch
                for pid in datasetPidsFromRorBasedSearch)

            # Add row to searchResultComparisonDict
            newRow = {
                'affiliation_name': authorAffiliationName,
                'affiliation_ror_urls': authorAffiliationRorUrlList,
                'nameBasedSearchUrl': nameBasedSearchUrl,
                'nameBasedSearchCount': len(datasetPidsFromNameBasedSearch),
                'rorBasedSearchUrl': rorBasedSearchUrl,
                'rorBasedSearchCount': len(datasetPidsFromRorBasedSearch),
                'name_based_search_items_include_all_ror_based_search_items': nameBasedSearchItemsIncludeAllRorBasedSearchItems
            }
            searchResultComparisonDict.append(newRow)

        # Export summary
        searchResultComparisonDf = pd.DataFrame(searchResultComparisonDict)
        filePath = os.path.join(installationDirPath,  'search_results_comparison.csv')
        searchResultComparisonDf.to_csv(filePath, index=False,  encoding='utf-8-sig')

            

100%|██████████| 14/14 [Elapsed:06:02<Remaining:00:0025.87s/it, DANS Data Station Social Sciences and Humanities: Wrocław University of Environmental and Life Sciences] 


In [ ]:
searchResultComparisonDf = pd.DataFrame(searchResultComparisonDict)
filePath = os.path.join(installationDirPath,  'search_results_comparison.csv')
searchResultComparisonDf.to_csv(filePath, index=False,  encoding='utf-8-sig')